# Cell 1 - Paired Binary 3D Voxel Occupancy Inspection Notebook

This notebook compares two audio files by converting each one into a binary 3D occupancy volume with axes `frequency band x quantized intensity bin x time frame`. The goal is to inspect whether the same voxelization pipeline produces stable, comparable structure across two machine sounds.

This `v0.5` notebook voxelizes the full loaded portion of both input files, computes whole-file surface and voxel-level outputs, and compares file B against file A across the entire available frame range.

The 3D plots show occupied voxels in a constructed representation, not waveform amplitude and not feature-channel geometry.

The dense spectral views, the voxel volumes, the surface extraction, the aligned difference tables, and the exported CSVs all operate over the full loaded clip, subject only to `MAX_AUDIO_SECONDS` if you set it.

That means the plotted tensor has shape `(NUM_BANDS, ENERGY_BINS, num_frames_full_clip)`. With the current defaults, `HOP_LENGTH = 256` and `TARGET_SR = 16000`, each frame spans `256 / 16000 = 0.016` seconds, so the time axis covers the whole loaded clip rather than one selected chunk.

The notebook exports full occupied-voxel CSVs for each file, full surface CSVs for each file, full aligned surface-difference CSVs, full occupied-voxel-difference CSVs, and summary CSVs so the complete processed output is available outside the notebook. If the files are long, those CSVs can be large, so use `MAX_AUDIO_SECONDS` when you need to constrain runtime or export size.


In [ ]:
# Cell 2 - Imports

from datetime import datetime
from pathlib import Path
import re

import librosa
import librosa.display
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy
import ipywidgets as widgets

from IPython.display import HTML, Markdown, display
from ipyfilechooser import FileChooser


In [ ]:
# Cell 3 - Configuration

# Audio loading
TARGET_SR = 16000
MAX_AUDIO_SECONDS = None
AUDIO_OFFSET_SECONDS = 0.0
MONO = True
AUDIO_FILE_PATH_A = None  # Optional fallback if chooser A does not render.
AUDIO_FILE_PATH_B = None  # Optional fallback if chooser B does not render.
FILE_CHOOSER_START_DIR = "datasets"

# Spectral frontend
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = "hann"

# Log-band projection
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0

# Energy scaling
LOG_POWER_REFERENCE = 1.0
EPSILON = 1e-8

# Normalization and quantization
NORMALIZE_BAND_ENERGY = True
NORMALIZATION_MODE = "per_clip_minmax"
ENERGY_BINS = 24

# Binary voxelization
VOXEL_MODE = "filled_column"   # or "one_hot"
LOW_ENERGY_FLOOR = 0.0
MIN_ACTIVE_BANDS_PER_FRAME = 0

# Plot controls
PLOT_MARKER_SIZE = 4
PLOT_MARKER_OPACITY = 0.75
PLOT_TEMPLATE = "plotly_white"
SHOW_TIME_AS_SECONDS = False
MAX_POINTS_PER_3D_PLOT = 150000
PRIMARY_INSPECTION_VIEW = "surface_only"

# Comparison labels
COMPARE_NAME_A = "a"
COMPARE_NAME_B = "b"

# Difference plot controls
PLOT_ONLY_CHANGED_DIFF = True
MIN_ABS_DELTA_TO_PLOT = 1
DIFF_PLOT_MARKER_SIZE = 5
DIFF_PLOT_MARKER_OPACITY = 0.9

# File support
SUPPORTED_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

# Optional debugging helpers
RUN_VOXEL_MODE_COMPARISON = False

# Export controls
EXPORT_TIMESTAMP_OVERRIDE = None  # Example: "20260317_101500"
EXPORT_SURFACE_CSV = True
EXPORT_SURFACE_DIFF_CSV = True
EXPORT_SUMMARY_CSV = True
SURFACE_DIFF_CSV_OUTPUT_PATH = None
SUMMARY_CSV_OUTPUT_PATH = None
COMPARISON_SUMMARY_CSV_OUTPUT_PATH = None

# Derived values
FILE_LABELS = ["a", "b"]


## Cell 4 - Parameter Guide

`NUM_BANDS` controls how finely frequency is sliced. `ENERGY_BINS` controls vertical resolution along the quantized intensity axis. `HOP_LENGTH` controls time resolution across the full loaded clip.

`VOXEL_MODE` controls whether each active `(band, frame)` becomes a single occupied intensity bin (`one_hot`) or a filled column from zero up to the active intensity (`filled_column`). `LOW_ENERGY_FLOOR` suppresses very weak normalized energy before voxelization.

This notebook compares two files in parallel using the surface-only view, which keeps only the top occupied voxel in each frequency-time column for display and CSV export.

`MAX_POINTS_PER_3D_PLOT` caps how many surface points each rendered 3D figure may display so long clips remain viewable. The surface CSV exports remain full-resolution even when the on-screen 3D plots are sampled.


In [ ]:
# Cell 5 - Dual File Chooser

display(Markdown("### Select two audio files to compare"))
display(Markdown(
    "Choose file `a` and file `b` below. If `ipyfilechooser` does not render in your frontend, set `AUDIO_FILE_PATH_A` and `AUDIO_FILE_PATH_B` in Cell 3 and rerun from Cell 7."
))

chooser_start_dir = Path(FILE_CHOOSER_START_DIR).expanduser()
if not chooser_start_dir.is_absolute():
    chooser_start_dir = (Path.cwd() / chooser_start_dir).resolve()
if not chooser_start_dir.exists():
    chooser_start_dir = Path.cwd()


def make_file_chooser(title, configured_path=None):
    chooser = FileChooser(path=str(chooser_start_dir))
    chooser.title = title
    chooser.show_hidden = False
    chooser.use_dir_icons = True
    chooser.filter_pattern = [f"*{ext}" for ext in SUPPORTED_EXTENSIONS]
    if configured_path:
        preset_path = Path(configured_path).expanduser()
        if not preset_path.is_absolute():
            preset_path = (Path.cwd() / preset_path).resolve()
        if preset_path.exists():
            chooser.reset(path=str(preset_path.parent), filename=preset_path.name)
    return chooser


file_chooser_a = make_file_chooser("Choose audio file a", AUDIO_FILE_PATH_A)
file_chooser_b = make_file_chooser("Choose audio file b", AUDIO_FILE_PATH_B)

selected_file_preview_a = widgets.HTML(value="<i>No file selected yet for a.</i>")
selected_file_preview_b = widgets.HTML(value="<i>No file selected yet for b.</i>")


def refresh_preview(chooser, preview_widget, configured_path, file_label):
    selected = getattr(chooser, "selected", None)
    if selected:
        preview_widget.value = f"<b>{file_label}:</b> {selected}"
    elif configured_path:
        preview_widget.value = f"<b>{file_label} configured path:</b> {configured_path}"
    else:
        preview_widget.value = f"<i>No file selected yet for {file_label}.</i>"


file_chooser_a.register_callback(lambda chooser: refresh_preview(chooser, selected_file_preview_a, AUDIO_FILE_PATH_A, "a"))
file_chooser_b.register_callback(lambda chooser: refresh_preview(chooser, selected_file_preview_b, AUDIO_FILE_PATH_B, "b"))
refresh_preview(file_chooser_a, selected_file_preview_a, AUDIO_FILE_PATH_A, "a")
refresh_preview(file_chooser_b, selected_file_preview_b, AUDIO_FILE_PATH_B, "b")

display(Markdown(f"Chooser start directory: `{chooser_start_dir}`"))
display(
    widgets.HBox(
        [
            widgets.VBox([widgets.HTML("<b>File a</b>"), file_chooser_a, selected_file_preview_a]),
            widgets.VBox([widgets.HTML("<b>File b</b>"), file_chooser_b, selected_file_preview_b]),
        ],
        layout=widgets.Layout(width="100%"),
    )
)


## Cell 6 - Load Note

The next cell loads exactly two selected audio files and prints summary information for both. The rest of the notebook keeps those two files aligned for direct visual comparison.


In [ ]:
# Cell 7 - Load Audio

def _resolve_audio_path(chooser, configured_path=None):
    """Resolve an audio path from the chooser selection or a config fallback."""
    if configured_path is not None and str(configured_path).strip():
        configured = Path(str(configured_path)).expanduser()
        if not configured.is_absolute():
            configured = (Path.cwd() / configured).resolve()
        return str(configured)

    selected = getattr(chooser, "selected", None)
    if selected:
        return str(Path(selected).expanduser())

    selected_path = getattr(chooser, "selected_path", None)
    selected_filename = getattr(chooser, "selected_filename", None)
    if selected_path and selected_filename:
        return str(Path(selected_path) / selected_filename)

    raise ValueError(
        "No file selected. Use Cell 5 to pick both audio files, or set AUDIO_FILE_PATH_A and AUDIO_FILE_PATH_B in Cell 3."
    )


def load_audio(file_path, target_sr, mono=True, offset_seconds=0.0, max_seconds=None):
    """Load one audio file with librosa and return waveform, sample rate, and metadata."""
    path = Path(file_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Selected file does not exist: {path}")

    if path.suffix.lower() not in {ext.lower() for ext in SUPPORTED_EXTENSIONS}:
        supported = ", ".join(SUPPORTED_EXTENSIONS)
        raise ValueError(f"Unsupported file extension '{path.suffix}'. Supported extensions: {supported}")

    y, sr = librosa.load(
        path,
        sr=target_sr,
        mono=mono,
        offset=offset_seconds,
        duration=max_seconds,
    )

    num_samples = y.shape[-1] if y.ndim > 1 else len(y)
    duration_seconds = num_samples / float(sr)
    channel_description = "mono" if y.ndim == 1 else f"{y.shape[0]} channels"
    metadata = {
        "file_path": str(path),
        "sample_rate": sr,
        "duration_seconds": duration_seconds,
        "num_samples": num_samples,
        "channel_description": channel_description,
        "mono_requested": mono,
    }
    return y, sr, metadata


audio_inputs = {
    "a": _resolve_audio_path(file_chooser_a, AUDIO_FILE_PATH_A),
    "b": _resolve_audio_path(file_chooser_b, AUDIO_FILE_PATH_B),
}
audio_results = {}
for file_label, audio_path in audio_inputs.items():
    y, sr, metadata = load_audio(
        audio_path,
        target_sr=TARGET_SR,
        mono=MONO,
        offset_seconds=AUDIO_OFFSET_SECONDS,
        max_seconds=MAX_AUDIO_SECONDS,
    )
    audio_results[file_label] = {
        "audio_path": audio_path,
        "y": y,
        "sr": sr,
        "audio_duration_seconds": metadata["duration_seconds"],
        "audio_metadata": metadata,
    }
    print(f"[{file_label}] File path: {metadata['file_path']}")
    print(f"[{file_label}] Sample rate: {metadata['sample_rate']} Hz")
    print(f"[{file_label}] Duration: {metadata['duration_seconds']:.3f} s")
    print(f"[{file_label}] Number of samples: {metadata['num_samples']}")
    print(f"[{file_label}] Mono/stereo result: {metadata['channel_description']}")
    print()


## Cell 8 - Spectral Frontend Note

The next step computes the same STFT-based spectral frontend for both files and projects each one into log-frequency bands. This gives a more stable basis than the raw waveform for paired voxel comparison.


In [ ]:
# Cell 9 - Spectral Frontend

def compute_stft_magnitude(y, n_fft, hop_length, win_length, window):
    """Compute an STFT magnitude spectrogram with shape (freq_bins, num_frames)."""
    y_mono = librosa.to_mono(y) if getattr(y, "ndim", 1) > 1 else y
    stft_complex = librosa.stft(
        y_mono,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
    )
    return np.abs(stft_complex)


def build_log_band_filters(sr, n_fft, num_bands, fmin, fmax):
    """Build a simple triangular filterbank with log-spaced center frequencies."""
    fft_freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    valid_fmax = min(float(fmax), float(sr) / 2.0)
    valid_fmin = max(float(fmin), fft_freqs[1] if len(fft_freqs) > 1 else float(fmin))
    if valid_fmin >= valid_fmax:
        raise ValueError("fmin must be smaller than fmax after accounting for the sample rate.")

    centers = np.geomspace(valid_fmin, valid_fmax, num=num_bands)
    boundaries = np.empty(num_bands + 1, dtype=float)
    boundaries[0] = valid_fmin
    boundaries[-1] = valid_fmax
    if num_bands > 1:
        boundaries[1:-1] = np.sqrt(centers[:-1] * centers[1:])

    filters = np.zeros((num_bands, len(fft_freqs)), dtype=float)
    for band_idx in range(num_bands):
        left = boundaries[band_idx]
        center = centers[band_idx]
        right = boundaries[band_idx + 1]
        left_mask = (fft_freqs >= left) & (fft_freqs <= center)
        right_mask = (fft_freqs >= center) & (fft_freqs <= right)
        if center > left:
            filters[band_idx, left_mask] = (fft_freqs[left_mask] - left) / (center - left)
        if right > center:
            filters[band_idx, right_mask] = (right - fft_freqs[right_mask]) / (right - center)
        band_sum = filters[band_idx].sum()
        if band_sum > 0:
            filters[band_idx] /= band_sum
    return filters


def project_to_log_bands(stft_mag, band_filters, eps=1e-8):
    """Project STFT magnitude into log-band energy with shape (num_bands, num_frames)."""
    power_spectrogram = stft_mag ** 2
    band_energy = band_filters @ power_spectrogram
    return np.maximum(band_energy, eps)


def compute_log_band_energy(y, sr, n_fft, hop_length, win_length, window, num_bands, fmin, fmax, eps=1e-8):
    """Compute STFT magnitude, log-band filters, and band energy for one waveform."""
    stft_mag = compute_stft_magnitude(y, n_fft, hop_length, win_length, window)
    band_filters = build_log_band_filters(sr, n_fft, num_bands, fmin, fmax)
    band_energy = project_to_log_bands(stft_mag, band_filters, eps=eps)
    return stft_mag, band_filters, band_energy


for file_label, result in audio_results.items():
    stft_mag, band_filters, band_energy = compute_log_band_energy(
        result["y"],
        result["sr"],
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW,
        num_bands=NUM_BANDS,
        fmin=FMIN,
        fmax=FMAX,
        eps=EPSILON,
    )
    result["stft_mag"] = stft_mag
    result["band_filters"] = band_filters
    result["band_energy"] = band_energy
    print(f"[{file_label}] STFT magnitude shape: {stft_mag.shape} (freq_bins, num_frames)")
    print(f"[{file_label}] Band filters shape: {band_filters.shape} (num_bands, freq_bins)")
    print(f"[{file_label}] Band energy shape: {band_energy.shape} (num_bands, num_frames)")
    print()


In [ ]:
# Cell 10 - Plot Waveforms In Parallel

def plotly_html_fragment(fig, include_plotlyjs=False):
    """Convert a Plotly figure into embeddable HTML."""
    return fig.to_html(full_html=False, include_plotlyjs=include_plotlyjs, config={"responsive": True})


def display_plotly_figure(fig):
    """Display one Plotly figure as inline HTML."""
    display(HTML(plotly_html_fragment(fig, include_plotlyjs=True)))


def display_plotly_figures_side_by_side(fig_a, fig_b, label_a="a", label_b="b"):
    """Display two Plotly figures individually, in sequence, as self-contained HTML blocks."""
    display(Markdown(f"### Plot {label_a}"))
    display_plotly_figure(fig_a)
    display(Markdown(f"### Plot {label_b}"))
    display_plotly_figure(fig_b)


def plot_waveform(y, sr, audio_path, template="plotly_white"):
    """Create a waveform figure with time in seconds on the x-axis."""
    y_plot = librosa.to_mono(y) if getattr(y, "ndim", 1) > 1 else y
    time_axis = np.arange(len(y_plot)) / float(sr)
    title_name = Path(audio_path).name
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time_axis, y=y_plot, mode="lines", name="waveform", line=dict(width=1.0)))
    fig.update_layout(
        template=template,
        title=f"Waveform: {title_name}",
        xaxis_title="Time (seconds)",
        yaxis_title="Amplitude",
        height=350,
    )
    return fig


waveform_fig_a = plot_waveform(audio_results["a"]["y"], audio_results["a"]["sr"], audio_results["a"]["audio_path"], template=PLOT_TEMPLATE)
waveform_fig_b = plot_waveform(audio_results["b"]["y"], audio_results["b"]["sr"], audio_results["b"]["audio_path"], template=PLOT_TEMPLATE)
display_plotly_figures_side_by_side(waveform_fig_a, waveform_fig_b, label_a="a", label_b="b")


In [ ]:
# Cell 11 - Plot Band Energy Heatmaps In Parallel

def plot_band_energy_heatmap(band_energy, sr, hop_length, title_name, template="plotly_white", show_time_as_seconds=False):
    """Create a dense log-band energy heatmap."""
    num_frames = band_energy.shape[1]
    x_values = np.arange(num_frames) * hop_length / float(sr) if show_time_as_seconds else np.arange(num_frames)
    x_title = "Time (seconds)" if show_time_as_seconds else "Frame index"
    z_values = np.log10(band_energy + EPSILON)
    fig = go.Figure(
        data=go.Heatmap(
            z=z_values,
            x=x_values,
            y=np.arange(band_energy.shape[0]),
            colorscale="Viridis",
            colorbar=dict(title="log10 energy"),
        )
    )
    fig.update_layout(
        template=template,
        title=f"Dense log-band energy: {title_name}",
        xaxis_title=x_title,
        yaxis_title="Frequency band index",
        height=450,
    )
    return fig


band_energy_fig_a = plot_band_energy_heatmap(
    audio_results["a"]["band_energy"],
    audio_results["a"]["sr"],
    HOP_LENGTH,
    Path(audio_results["a"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
band_energy_fig_b = plot_band_energy_heatmap(
    audio_results["b"]["band_energy"],
    audio_results["b"]["sr"],
    HOP_LENGTH,
    Path(audio_results["b"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
display_plotly_figures_side_by_side(band_energy_fig_a, band_energy_fig_b, label_a="a", label_b="b")


## Cell 12 - Normalization And Quantization Note

The next step takes the per-band energy values for both files and maps them into a finite number of intensity bins so each `(time, frequency)` location can be voxelized into the 3D occupancy volume. The goal is structural coherence and side-by-side comparability, not audio fidelity.


In [ ]:
# Cell 13 - Normalize And Quantize Both Files

def log_compress_band_energy(band_energy, eps=1e-8, reference=1.0):
    """Apply a readable log compression to band energy."""
    safe_reference = max(float(reference), eps)
    return np.log((band_energy / safe_reference) + eps)


def normalize_band_energy(log_band_energy, mode="per_clip_minmax", eps=1e-8):
    """Normalize log-band energy into [0, 1]."""
    if mode == "per_clip_minmax":
        x_min = np.min(log_band_energy)
        x_max = np.max(log_band_energy)
        denom = max(x_max - x_min, eps)
        return np.clip((log_band_energy - x_min) / denom, 0.0, 1.0)
    raise ValueError(f"Unsupported normalization mode: {mode}")


def quantize_to_energy_bins(normalized_band_energy, energy_bins, floor=0.0):
    """Quantize normalized band energy into integer bins and return an active mask."""
    if energy_bins <= 0:
        raise ValueError("energy_bins must be positive.")
    clipped = np.clip(normalized_band_energy, 0.0, 1.0)
    active_mask = clipped > float(floor)
    energy_bin_indices = np.floor(clipped * energy_bins).astype(int)
    energy_bin_indices = np.clip(energy_bin_indices, 0, energy_bins - 1)
    energy_bin_indices = np.where(active_mask, energy_bin_indices, 0)
    return energy_bin_indices, active_mask


for file_label, result in audio_results.items():
    log_band_energy = log_compress_band_energy(result["band_energy"], eps=EPSILON, reference=LOG_POWER_REFERENCE)
    normalized_band_energy = (
        normalize_band_energy(log_band_energy, mode=NORMALIZATION_MODE, eps=EPSILON)
        if NORMALIZE_BAND_ENERGY
        else log_band_energy.copy()
    )
    energy_bin_indices, active_mask = quantize_to_energy_bins(
        normalized_band_energy,
        energy_bins=ENERGY_BINS,
        floor=LOW_ENERGY_FLOOR,
    )

    if MIN_ACTIVE_BANDS_PER_FRAME > 0:
        active_counts_per_frame = active_mask.sum(axis=0)
        valid_frames = active_counts_per_frame >= MIN_ACTIVE_BANDS_PER_FRAME
        active_mask = active_mask & valid_frames[np.newaxis, :]
        energy_bin_indices = np.where(active_mask, energy_bin_indices, 0)
        print(f"[{file_label}] Frames meeting MIN_ACTIVE_BANDS_PER_FRAME: {int(valid_frames.sum())} / {len(valid_frames)}")

    result["log_band_energy"] = log_band_energy
    result["normalized_band_energy"] = normalized_band_energy
    result["energy_bin_indices"] = energy_bin_indices
    result["active_mask"] = active_mask
    print(f"[{file_label}] Log band energy shape: {log_band_energy.shape}")
    print(f"[{file_label}] Normalized band energy shape: {normalized_band_energy.shape}")
    print(f"[{file_label}] Energy bin indices shape: {energy_bin_indices.shape}")
    print(f"[{file_label}] Active mask shape: {active_mask.shape}")
    print(f"[{file_label}] Normalized min/max: {normalized_band_energy.min():.4f} / {normalized_band_energy.max():.4f}")
    print(f"[{file_label}] Active cells: {int(active_mask.sum())} / {active_mask.size}")
    print()


In [ ]:
# Cell 14 - Plot Normalized Band Energy In Parallel

def plot_normalized_band_energy_heatmap(normalized_band_energy, sr, hop_length, title_name, template="plotly_white", show_time_as_seconds=False):
    """Create a normalized band energy heatmap."""
    num_frames = normalized_band_energy.shape[1]
    x_values = np.arange(num_frames) * hop_length / float(sr) if show_time_as_seconds else np.arange(num_frames)
    x_title = "Time (seconds)" if show_time_as_seconds else "Frame index"
    fig = go.Figure(
        data=go.Heatmap(
            z=normalized_band_energy,
            x=x_values,
            y=np.arange(normalized_band_energy.shape[0]),
            colorscale="Viridis",
            zmin=0.0,
            zmax=1.0,
            colorbar=dict(title="normalized energy"),
        )
    )
    fig.update_layout(
        template=template,
        title=f"Normalized band energy: {title_name}",
        xaxis_title=x_title,
        yaxis_title="Frequency band index",
        height=450,
    )
    return fig


normalized_energy_fig_a = plot_normalized_band_energy_heatmap(
    audio_results["a"]["normalized_band_energy"],
    audio_results["a"]["sr"],
    HOP_LENGTH,
    Path(audio_results["a"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
normalized_energy_fig_b = plot_normalized_band_energy_heatmap(
    audio_results["b"]["normalized_band_energy"],
    audio_results["b"]["sr"],
    HOP_LENGTH,
    Path(audio_results["b"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
display_plotly_figures_side_by_side(normalized_energy_fig_a, normalized_energy_fig_b, label_a="a", label_b="b")


In [ ]:
# Cell 15 - Plot Quantized Energy Bins In Parallel

def plot_energy_bin_heatmap(energy_bin_indices, active_mask, sr, hop_length, title_name, template="plotly_white", show_time_as_seconds=False):
    """Create a quantized energy bin heatmap."""
    num_frames = energy_bin_indices.shape[1]
    x_values = np.arange(num_frames) * hop_length / float(sr) if show_time_as_seconds else np.arange(num_frames)
    x_title = "Time (seconds)" if show_time_as_seconds else "Frame index"
    z_values = np.where(active_mask, energy_bin_indices, np.nan)
    fig = go.Figure(
        data=go.Heatmap(
            z=z_values,
            x=x_values,
            y=np.arange(energy_bin_indices.shape[0]),
            colorscale="Turbo",
            zmin=0,
            zmax=max(ENERGY_BINS - 1, 0),
            colorbar=dict(title="energy bin"),
        )
    )
    fig.update_layout(
        template=template,
        title=f"Quantized energy bins: {title_name}",
        xaxis_title=x_title,
        yaxis_title="Frequency band index",
        height=450,
    )
    return fig


energy_bin_fig_a = plot_energy_bin_heatmap(
    audio_results["a"]["energy_bin_indices"],
    audio_results["a"]["active_mask"],
    audio_results["a"]["sr"],
    HOP_LENGTH,
    Path(audio_results["a"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
energy_bin_fig_b = plot_energy_bin_heatmap(
    audio_results["b"]["energy_bin_indices"],
    audio_results["b"]["active_mask"],
    audio_results["b"]["sr"],
    HOP_LENGTH,
    Path(audio_results["b"]["audio_path"]).name,
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
display_plotly_figures_side_by_side(energy_bin_fig_a, energy_bin_fig_b, label_a="a", label_b="b")


## Cell 16 - Full-Clip Voxelization Note

The next cells use the full quantized band-energy matrices for both files and convert them into binary voxel occupancy volumes. In `one_hot` mode, each active `(band, frame)` creates exactly one occupied voxel. In `filled_column` mode, every voxel from zero up to the active intensity bin is occupied, which often creates thicker and more stable shapes.


In [ ]:
# Cell 17 - Prepare Full-Clip Matrices

def prepare_full_clip_arrays(energy_bin_indices, active_mask):
    """Validate and return full-clip arrays with shape (num_bands, num_frames)."""
    if energy_bin_indices.shape != active_mask.shape:
        raise ValueError("energy_bin_indices and active_mask must have matching shapes.")
    if energy_bin_indices.ndim != 2:
        raise ValueError(
            f"Expected 2D arrays with shape (num_bands, num_frames). Got shape {energy_bin_indices.shape}."
        )
    if energy_bin_indices.shape[1] == 0:
        raise ValueError("No frames are available after spectral processing.")
    return energy_bin_indices, active_mask


for file_label, result in audio_results.items():
    full_energy_bins, full_active_mask = prepare_full_clip_arrays(
        result["energy_bin_indices"],
        result["active_mask"],
    )
    result["full_energy_bins"] = full_energy_bins
    result["full_active_mask"] = full_active_mask
    result["num_frames_full_clip"] = int(full_energy_bins.shape[1])
    print(f"[{file_label}] Full energy bins shape: {full_energy_bins.shape}")
    print(f"[{file_label}] Full active mask shape: {full_active_mask.shape}")
    print(f"[{file_label}] Full frame range: [0, {result['num_frames_full_clip'] - 1}]")
    print()


In [ ]:
# Cell 18 - Build Voxel Volumes

def build_binary_voxel_volume(full_energy_bins, full_active_mask, energy_bins, voxel_mode="filled_column"):
    """Build a binary voxel volume with shape (num_bands, energy_bins, num_frames).

    Axis order is fixed:
    - axis 0 = frequency band
    - axis 1 = quantized intensity bin
    - axis 2 = frame index
    """
    if full_energy_bins.shape != full_active_mask.shape:
        raise ValueError("full_energy_bins and full_active_mask must have matching shapes.")
    if voxel_mode not in {"one_hot", "filled_column"}:
        raise ValueError("voxel_mode must be 'one_hot' or 'filled_column'.")
    if energy_bins <= 0:
        raise ValueError("energy_bins must be positive.")

    full_energy_bins = np.clip(full_energy_bins.astype(int), 0, energy_bins - 1)
    full_active_mask = full_active_mask.astype(bool)
    num_bands, num_frames = full_energy_bins.shape

    if voxel_mode == "one_hot":
        voxel_volume = np.zeros((num_bands, energy_bins, num_frames), dtype=bool)
        band_indices, frame_indices = np.nonzero(full_active_mask)
        voxel_volume[
            band_indices,
            full_energy_bins[band_indices, frame_indices],
            frame_indices,
        ] = True
        return voxel_volume

    energy_axis = np.arange(energy_bins, dtype=int)[None, :, None]
    voxel_volume = full_active_mask[:, None, :] & (energy_axis <= full_energy_bins[:, None, :])
    return voxel_volume


for file_label, result in audio_results.items():
    voxel_volume = build_binary_voxel_volume(
        result["full_energy_bins"],
        result["full_active_mask"],
        energy_bins=ENERGY_BINS,
        voxel_mode=VOXEL_MODE,
    )
    result["voxel_volume"] = voxel_volume
    result["total_occupied_voxels"] = int(voxel_volume.sum())
    result["occupancy_ratio"] = result["total_occupied_voxels"] / float(voxel_volume.size)
    print(f"[{file_label}] Voxel volume shape: {voxel_volume.shape}")
    print(f"[{file_label}] Voxel volume dtype: {voxel_volume.dtype}")
    print(f"[{file_label}] Total occupied voxels: {result['total_occupied_voxels']}")
    print(f"[{file_label}] Occupancy ratio: {result['occupancy_ratio']:.6f}")
    print()


## Cell 19 - Surface View Explanation

The next inspection view keeps only the highest occupied voxel in each frequency-time column. For each `(band_idx, frame_idx)` column, it retains the top surface voxel and discards the lower filled voxels for visualization.

This surface-only view is usually easier to interpret than the fully filled volume because it reduces clutter while still showing rails, bursts, sloped traces, and repeating structure. The full occupied-voxel plot remains available later as an optional debugging view.


In [ ]:
# Cell 20 - Surface-Only Preparation Summary

for file_label, result in audio_results.items():
    num_bands, _, num_frames = result["voxel_volume"].shape
    print(f"[{file_label}] Full voxel volume ready for surface extraction.")
    print(f"[{file_label}] Num bands: {num_bands}")
    print(f"[{file_label}] Num frames: {num_frames}")
    print()


In [ ]:
# Cell 21 - Plot Full-Clip Surface Voxels In Parallel And Export CSVs

def resolve_voxel_output_dir():
    """Resolve the notebook output directory in a way that works from common Jupyter working directories."""
    candidate_dirs = [Path("preprocessing/02.preprocessing-display/output"), Path("output")]
    for candidate in candidate_dirs:
        resolved = (Path.cwd() / candidate).resolve()
        if resolved.exists():
            return resolved
    resolved = (Path.cwd() / candidate_dirs[0]).resolve()
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved


def sanitize_filename_stem(file_path):
    """Convert a source filename stem into a filesystem-safe token."""
    stem = Path(file_path).stem
    stem = re.sub(r"[^A-Za-z0-9._-]+", "-", stem).strip("-._")
    return stem or "audio"


def build_export_timestamp(override=None):
    """Build the timestamp prefix used for CSV exports."""
    if override is not None and str(override).strip():
        return str(override).strip()
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def resolve_compare_name(file_label):
    """Map internal file labels to human-readable comparison labels."""
    if file_label == "a":
        return COMPARE_NAME_A
    if file_label == "b":
        return COMPARE_NAME_B
    return str(file_label)


def resolve_csv_export_path(default_filename, output_path=None):
    """Resolve a CSV export path inside the notebook output directory by default."""
    output_dir = resolve_voxel_output_dir()
    if output_path is None or not str(output_path).strip():
        export_path = output_dir / default_filename
    else:
        export_path = Path(output_path)
        if not export_path.is_absolute():
            export_path = output_dir / export_path
        if export_path.suffix.lower() != ".csv":
            export_path = export_path.with_suffix(".csv")
    export_path.parent.mkdir(parents=True, exist_ok=True)
    return export_path


def extract_surface_voxels(voxel_volume, voxel_mode):
    """Extract the highest occupied voxel in each (band_idx, frame_idx) column."""
    if voxel_volume.ndim != 3:
        raise ValueError(
            f"voxel_volume must be 3D with shape (num_bands, energy_bins, num_frames). Got shape {voxel_volume.shape}."
        )
    occupied_columns = voxel_volume.any(axis=1)
    band_indices, frame_indices = np.nonzero(occupied_columns)
    if band_indices.size == 0:
        return pd.DataFrame(columns=["band_idx", "energy_bin", "frame_idx", "is_surface", "voxel_mode"])

    top_energy_bins = voxel_volume.shape[1] - 1 - np.argmax(voxel_volume[:, ::-1, :], axis=1)
    surface_voxels_df = pd.DataFrame(
        {
            "band_idx": band_indices.astype(int),
            "energy_bin": top_energy_bins[band_indices, frame_indices].astype(int),
            "frame_idx": frame_indices.astype(int),
            "is_surface": True,
            "voxel_mode": voxel_mode,
        }
    )
    return surface_voxels_df.sort_values(["frame_idx", "band_idx"], kind="stable").reset_index(drop=True)


def build_surface_export_df(surface_voxels_df, result, file_label, compare_name):
    """Attach export metadata to the extracted surface dataframe."""
    export_df = surface_voxels_df.copy()
    export_df["frame_time_seconds"] = export_df["frame_idx"] * HOP_LENGTH / float(result["sr"])
    export_df["color_value"] = export_df["energy_bin"]
    export_df["audio_path"] = str(Path(result["audio_path"]).resolve())
    export_df["file_label"] = file_label
    export_df["compare_name"] = compare_name
    export_df["title"] = f"3D voxel surface full volume ({VOXEL_MODE})"
    export_df["x_axis"] = "band_idx"
    export_df["y_axis"] = "energy_bin"
    export_df["z_axis"] = "frame_idx"
    export_df["colorscale"] = "Turbo"
    export_df["marker_size"] = max(PLOT_MARKER_SIZE, 5)
    export_df["marker_opacity"] = max(PLOT_MARKER_OPACITY, 0.9)
    export_df["template"] = PLOT_TEMPLATE
    export_df["frame_duration_seconds"] = HOP_LENGTH / float(result["sr"])
    export_df["num_frames_full_clip"] = result["num_frames_full_clip"]
    return export_df


def export_dataframe_csv(dataframe, export_path):
    """Write a dataframe to CSV and return the resolved path."""
    dataframe.to_csv(export_path, index=False)
    return export_path


def cap_3d_plot_points(plot_df, max_points, sort_columns):
    """Cap the number of rendered surface points in a 3D plot without changing the exported CSV data."""
    total_points = len(plot_df)
    if total_points == 0:
        return plot_df.copy(), total_points, False
    if max_points is None or int(max_points) <= 0 or total_points <= int(max_points):
        return plot_df.copy(), total_points, False

    ordered_df = plot_df.sort_values(sort_columns, kind="stable").reset_index(drop=True)
    sample_indices = np.linspace(0, total_points - 1, num=int(max_points), dtype=int)
    sample_indices = np.unique(sample_indices)
    sampled_df = ordered_df.iloc[sample_indices].reset_index(drop=True)
    return sampled_df, total_points, True


def plot_surface_voxels_3d(
    surface_voxels_df,
    title_suffix,
    marker_size=5,
    marker_opacity=0.9,
    template="plotly_white",
    max_points=None,
):
    """Plot only the top surface voxels in 3D."""
    if surface_voxels_df.empty:
        raise ValueError("Surface voxel dataframe is empty. There is nothing to plot.")

    plot_df, total_points, was_sampled = cap_3d_plot_points(
        surface_voxels_df,
        max_points=max_points,
        sort_columns=["frame_idx", "band_idx", "energy_bin"],
    )
    sample_note = (
        f"sampled {len(plot_df):,} of {total_points:,} surface voxels"
        if was_sampled
        else f"{total_points:,} surface voxels"
    )
    voxel_mode_label = surface_voxels_df["voxel_mode"].iloc[0]
    customdata = np.column_stack(
        [
            plot_df["band_idx"].to_numpy(),
            plot_df["energy_bin"].to_numpy(),
            plot_df["frame_idx"].to_numpy(),
            plot_df["frame_time_seconds"].to_numpy(),
        ]
    )
    fig = go.Figure(
        data=go.Scatter3d(
            x=plot_df["band_idx"],
            y=plot_df["energy_bin"],
            z=plot_df["frame_idx"],
            mode="markers",
            marker=dict(
                size=marker_size,
                opacity=marker_opacity,
                color=plot_df["energy_bin"],
                colorscale="Turbo",
                colorbar=dict(title="energy bin"),
            ),
            customdata=customdata,
            hovertemplate=(
                "Frequency band: %{customdata[0]}<br>"
                "Energy bin: %{customdata[1]}<br>"
                "Frame index: %{customdata[2]}<br>"
                "Time (seconds): %{customdata[3]:.3f}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        template=template,
        title=f"3D voxel surface-only occupancy full volume ({voxel_mode_label}, {sample_note}): {title_suffix}",
        scene=dict(
            xaxis_title="Frequency band index",
            yaxis_title="Quantized intensity bin",
            zaxis_title="Frame index across loaded clip",
        ),
        height=700,
    )
    return fig


export_timestamp = build_export_timestamp(EXPORT_TIMESTAMP_OVERRIDE)
for file_label, result in audio_results.items():
    compare_name = resolve_compare_name(file_label)

    surface_voxels_df = extract_surface_voxels(result["voxel_volume"], VOXEL_MODE)
    if surface_voxels_df.empty:
        raise ValueError(f"No surface voxels were extracted for {compare_name}.")
    surface_voxels_df = build_surface_export_df(
        surface_voxels_df,
        result,
        file_label=file_label,
        compare_name=compare_name,
    )
    result["surface_voxels_df"] = surface_voxels_df
    result["total_surface_voxels"] = len(surface_voxels_df)

    safe_stem = sanitize_filename_stem(result["audio_path"])
    result["surface_voxel_csv_path"] = None
    if EXPORT_SURFACE_CSV:
        surface_export_path = resolve_csv_export_path(
            f"{export_timestamp}_{file_label}_{safe_stem}_surface_voxels.csv"
        )
        result["surface_voxel_csv_path"] = export_dataframe_csv(surface_voxels_df, surface_export_path)

    print(f"[{compare_name}] Surface voxel rows: {len(surface_voxels_df)}")
    print(f"[{compare_name}] Maximum possible surface rows: {result['voxel_volume'].shape[0] * result['voxel_volume'].shape[2]}")
    if MAX_POINTS_PER_3D_PLOT is None or int(MAX_POINTS_PER_3D_PLOT) <= 0:
        print(f"[{compare_name}] Surface 3D plots are uncapped.")
    else:
        print(f"[{compare_name}] Surface 3D plot point cap: {int(MAX_POINTS_PER_3D_PLOT):,}")
    if result["surface_voxel_csv_path"] is not None:
        print(f"[{compare_name}] Saved surface voxel CSV to: {result['surface_voxel_csv_path']}")
    else:
        print(f"[{compare_name}] EXPORT_SURFACE_CSV is False; skipping surface voxel CSV export.")

    result["surface_voxel_fig"] = plot_surface_voxels_3d(
        surface_voxels_df,
        title_suffix=f"{compare_name}: {Path(result['audio_path']).name}",
        marker_size=max(PLOT_MARKER_SIZE, 5),
        marker_opacity=max(PLOT_MARKER_OPACITY, 0.9),
        template=PLOT_TEMPLATE,
        max_points=MAX_POINTS_PER_3D_PLOT,
    )
    print()


display(Markdown("## Full surface 3D plots"))
display_plotly_figures_side_by_side(
    audio_results["a"]["surface_voxel_fig"],
    audio_results["b"]["surface_voxel_fig"],
    label_a=COMPARE_NAME_A,
    label_b=COMPARE_NAME_B,
)


In [ ]:
# Cell 22 - Plot Surface Projections In Parallel

def plot_surface_projection_frequency_time(surface_voxels_df, num_bands, num_frames, sr, template="plotly_white", show_time_as_seconds=False):
    """Plot the extracted surface as a 2D heatmap."""
    if surface_voxels_df.empty:
        raise ValueError("Surface voxel dataframe is empty. There is nothing to project.")
    projection = np.full((num_bands, num_frames), np.nan, dtype=float)
    projection[
        surface_voxels_df["band_idx"].to_numpy(dtype=int),
        surface_voxels_df["frame_idx"].to_numpy(dtype=int),
    ] = surface_voxels_df["energy_bin"].to_numpy(dtype=float)
    if show_time_as_seconds:
        x_values = np.arange(num_frames) * HOP_LENGTH / float(sr)
        x_title = "Time (seconds)"
    else:
        x_values = np.arange(num_frames)
        x_title = "Frame index"
    fig = go.Figure(
        data=go.Heatmap(
            z=projection,
            x=x_values,
            y=np.arange(num_bands),
            colorscale="Turbo",
            zmin=0,
            zmax=max(ENERGY_BINS - 1, 0),
            colorbar=dict(title="surface energy bin"),
        )
    )
    fig.update_layout(
        template=template,
        title="Surface projection over frequency and time",
        xaxis_title=x_title,
        yaxis_title="Frequency band index",
        height=450,
    )
    return fig


surface_projection_fig_a = plot_surface_projection_frequency_time(
    audio_results["a"]["surface_voxels_df"],
    num_bands=audio_results["a"]["voxel_volume"].shape[0],
    num_frames=audio_results["a"]["voxel_volume"].shape[2],
    sr=audio_results["a"]["sr"],
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
surface_projection_fig_b = plot_surface_projection_frequency_time(
    audio_results["b"]["surface_voxels_df"],
    num_bands=audio_results["b"]["voxel_volume"].shape[0],
    num_frames=audio_results["b"]["voxel_volume"].shape[2],
    sr=audio_results["b"]["sr"],
    template=PLOT_TEMPLATE,
    show_time_as_seconds=SHOW_TIME_AS_SECONDS,
)
display_plotly_figures_side_by_side(
    surface_projection_fig_a,
    surface_projection_fig_b,
    label_a=COMPARE_NAME_A,
    label_b=COMPARE_NAME_B,
)


## Cell 23 - Difference Surface View

The two surface-only plots above show the absolute top-surface voxel geometry for file A and file B independently.

The next plot aligns both surfaces on `(band_idx, frame_idx)` and visualizes the signed difference surface using `delta_energy_bin = energy_bin_b - energy_bin_a`. Positive deltas mean file B is higher than file A, and negative deltas mean file B is lower than file A.

This is a deviation view rather than the original energy-bin geometry, so it is useful for spotting where the two surfaces separate, collapse, or shift over time across the whole loaded clip.


In [ ]:
# Cell 24 - Surface Difference And Optional Surface-Mode Views

def build_surface_difference_df(surface_a_df, surface_b_df, name_a="a", name_b="b"):
    """
    Align two surface dataframes on (band_idx, frame_idx) and compute
    signed and absolute energy-bin differences.
    """
    required_columns = {"band_idx", "frame_idx", "energy_bin"}
    missing_a = required_columns.difference(surface_a_df.columns)
    missing_b = required_columns.difference(surface_b_df.columns)
    if missing_a:
        raise ValueError(f"surface_a_df is missing required columns: {sorted(missing_a)}")
    if missing_b:
        raise ValueError(f"surface_b_df is missing required columns: {sorted(missing_b)}")
    if surface_a_df.duplicated(["band_idx", "frame_idx"]).any():
        raise ValueError(f"surface_a_df contains duplicate (band_idx, frame_idx) rows for {name_a}.")
    if surface_b_df.duplicated(["band_idx", "frame_idx"]).any():
        raise ValueError(f"surface_b_df contains duplicate (band_idx, frame_idx) rows for {name_b}.")

    columns_a = ["band_idx", "frame_idx", "energy_bin"]
    columns_b = ["band_idx", "frame_idx", "energy_bin"]
    rename_a = {"energy_bin": "energy_bin_a"}
    rename_b = {"energy_bin": "energy_bin_b"}

    if "audio_path" in surface_a_df.columns:
        columns_a.append("audio_path")
        rename_a["audio_path"] = "audio_path_a"
    if "audio_path" in surface_b_df.columns:
        columns_b.append("audio_path")
        rename_b["audio_path"] = "audio_path_b"

    aligned_a = surface_a_df.loc[:, columns_a].rename(columns=rename_a)
    aligned_b = surface_b_df.loc[:, columns_b].rename(columns=rename_b)

    diff_df = pd.merge(
        aligned_a,
        aligned_b,
        on=["band_idx", "frame_idx"],
        how="outer",
        sort=True,
    )
    diff_df["present_a"] = diff_df["energy_bin_a"].notna()
    diff_df["present_b"] = diff_df["energy_bin_b"].notna()
    diff_df["delta_energy_bin"] = diff_df["energy_bin_b"] - diff_df["energy_bin_a"]
    diff_df["abs_delta_energy_bin"] = diff_df["delta_energy_bin"].abs()
    diff_df["changed"] = (~diff_df["present_a"]) | (~diff_df["present_b"]) | (diff_df["delta_energy_bin"] != 0)
    diff_df["frame_time_seconds"] = diff_df["frame_idx"] * HOP_LENGTH / float(TARGET_SR)
    return diff_df.sort_values(["frame_idx", "band_idx"], kind="stable").reset_index(drop=True)


def plot_surface_difference_3d(
    diff_df,
    only_changed=True,
    min_abs_delta=1,
    marker_size=5,
    marker_opacity=0.9,
    template="plotly_white",
):
    """Plot the signed surface difference in 3D."""
    required_columns = {
        "band_idx",
        "frame_idx",
        "energy_bin_a",
        "energy_bin_b",
        "delta_energy_bin",
        "abs_delta_energy_bin",
        "changed",
        "frame_time_seconds",
    }
    missing_columns = required_columns.difference(diff_df.columns)
    if missing_columns:
        raise ValueError(f"diff_df is missing required columns: {sorted(missing_columns)}")

    plot_df = diff_df.copy()
    plot_df = plot_df[plot_df["delta_energy_bin"].notna()].copy()
    if only_changed:
        plot_df = plot_df[plot_df["changed"]].copy()
    plot_df = plot_df[plot_df["abs_delta_energy_bin"] >= float(min_abs_delta)].copy()

    title = f"Surface difference ({COMPARE_NAME_B} - {COMPARE_NAME_A})"
    if plot_df.empty:
        fig = go.Figure()
        fig.update_layout(
            template=template,
            title=f"{title}: no points meet the current filters",
            scene=dict(
                xaxis_title="Frequency band index",
                yaxis_title=f"Signed energy-bin difference ({COMPARE_NAME_B} - {COMPARE_NAME_A})",
                zaxis_title="Frame index across loaded clip",
            ),
            height=700,
        )
        fig.add_annotation(
            text="No aligned surface differences satisfy the current plotting filters.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )
        return fig

    max_abs_delta = float(plot_df["abs_delta_energy_bin"].max())
    color_limit = max(max_abs_delta, 1.0)
    customdata = np.column_stack(
        [
            plot_df["band_idx"].to_numpy(),
            plot_df["frame_idx"].to_numpy(),
            plot_df["frame_time_seconds"].to_numpy(),
            plot_df["energy_bin_a"].to_numpy(),
            plot_df["energy_bin_b"].to_numpy(),
            plot_df["delta_energy_bin"].to_numpy(),
            plot_df["abs_delta_energy_bin"].to_numpy(),
        ]
    )
    fig = go.Figure(
        data=go.Scatter3d(
            x=plot_df["band_idx"],
            y=plot_df["delta_energy_bin"],
            z=plot_df["frame_idx"],
            mode="markers",
            marker=dict(
                size=marker_size,
                opacity=marker_opacity,
                color=plot_df["delta_energy_bin"],
                colorscale="RdBu",
                cmin=-color_limit,
                cmax=color_limit,
                colorbar=dict(title=f"delta bin ({COMPARE_NAME_B} - {COMPARE_NAME_A})"),
            ),
            customdata=customdata,
            hovertemplate=(
                "Frequency band: %{customdata[0]}<br>"
                "Frame index: %{customdata[1]}<br>"
                "Time (seconds): %{customdata[2]:.3f}<br>"
                "Energy bin a: %{customdata[3]}<br>"
                "Energy bin b: %{customdata[4]}<br>"
                "Delta energy bin: %{customdata[5]}<br>"
                "Absolute delta: %{customdata[6]}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        template=template,
        title=f"Signed surface difference across the full loaded clip ({COMPARE_NAME_B} - {COMPARE_NAME_A})",
        scene=dict(
            xaxis_title="Frequency band index",
            yaxis_title=f"Signed energy-bin difference ({COMPARE_NAME_B} - {COMPARE_NAME_A})",
            zaxis_title="Frame index across loaded clip",
        ),
        height=700,
    )
    return fig


def export_surface_difference_csv(diff_df, output_path, audio_path_a, audio_path_b, export_timestamp):
    """Export the aligned surface comparison dataframe to CSV."""
    safe_stem_a = sanitize_filename_stem(audio_path_a)
    safe_stem_b = sanitize_filename_stem(audio_path_b)
    export_path = resolve_csv_export_path(
        f"{export_timestamp}_surface_diff_{safe_stem_a}_vs_{safe_stem_b}.csv",
        output_path=output_path,
    )
    diff_df.to_csv(export_path, index=False)
    print(f"Saved full surface difference CSV to: {export_path}")
    print(f"Aligned surface comparison rows exported: {len(diff_df)}")
    return export_path


surface_diff_df = build_surface_difference_df(
    audio_results["a"]["surface_voxels_df"],
    audio_results["b"]["surface_voxels_df"],
    name_a=COMPARE_NAME_A,
    name_b=COMPARE_NAME_B,
)
changed_surface_row_count = int(surface_diff_df["changed"].sum())
abs_delta_series = surface_diff_df["abs_delta_energy_bin"].dropna()
mean_abs_delta = float(abs_delta_series.mean()) if not abs_delta_series.empty else float("nan")
max_abs_delta = float(abs_delta_series.max()) if not abs_delta_series.empty else float("nan")

print(f"Aligned surface comparison rows: {len(surface_diff_df)}")
print(f"Changed surface rows: {changed_surface_row_count}")
print(f"Mean absolute surface delta: {mean_abs_delta:.3f}" if not np.isnan(mean_abs_delta) else "Mean absolute surface delta: nan")
print(f"Max absolute surface delta: {max_abs_delta:.3f}" if not np.isnan(max_abs_delta) else "Max absolute surface delta: nan")

surface_diff_csv_path = None
if EXPORT_SURFACE_DIFF_CSV:
    surface_diff_csv_path = export_surface_difference_csv(
        surface_diff_df,
        SURFACE_DIFF_CSV_OUTPUT_PATH,
        audio_results["a"]["audio_path"],
        audio_results["b"]["audio_path"],
        export_timestamp,
    )
else:
    print("EXPORT_SURFACE_DIFF_CSV is False; skipping full surface difference CSV export.")

surface_diff_fig = plot_surface_difference_3d(
    surface_diff_df,
    only_changed=PLOT_ONLY_CHANGED_DIFF,
    min_abs_delta=MIN_ABS_DELTA_TO_PLOT,
    marker_size=DIFF_PLOT_MARKER_SIZE,
    marker_opacity=DIFF_PLOT_MARKER_OPACITY,
    template=PLOT_TEMPLATE,
)
display(Markdown("### Difference plot"))
display_plotly_figure(surface_diff_fig)


def build_and_plot_both_voxel_modes(full_energy_bins, full_active_mask, energy_bins, title_prefix, marker_size=4, marker_opacity=0.75, template="plotly_white"):
    """Build both voxel modes and plot their surface-only views over the full loaded clip."""
    comparison_results = {}
    for voxel_mode in ["one_hot", "filled_column"]:
        comparison_volume = build_binary_voxel_volume(
            full_energy_bins,
            full_active_mask,
            energy_bins=energy_bins,
            voxel_mode=voxel_mode,
        )
        comparison_surface_df = extract_surface_voxels(comparison_volume, voxel_mode)
        if not comparison_surface_df.empty:
            comparison_surface_df["frame_time_seconds"] = comparison_surface_df["frame_idx"] * HOP_LENGTH / float(TARGET_SR)
        occupied = int(comparison_volume.sum())
        surface_points = len(comparison_surface_df)
        density = occupied / float(comparison_volume.size)
        comparison_results[voxel_mode] = {
            "voxel_volume": comparison_volume,
            "surface_voxels_df": comparison_surface_df,
            "occupied_voxels": occupied,
            "surface_voxels": surface_points,
            "occupancy_density": density,
        }
        print(
            f"{title_prefix} {voxel_mode}: occupied_voxels={occupied}, surface_voxels={surface_points}, occupancy_density={density:.6f}"
        )
        if comparison_surface_df.empty:
            print(f"{title_prefix} {voxel_mode}: no surface voxels to plot")
        else:
            comparison_results[voxel_mode]["surface_fig"] = plot_surface_voxels_3d(
                comparison_surface_df,
                title_suffix=f"{title_prefix} ({voxel_mode})",
                marker_size=max(marker_size, 5),
                marker_opacity=max(marker_opacity, 0.9),
                template=template,
                max_points=MAX_POINTS_PER_3D_PLOT,
            )
    return comparison_results


if RUN_VOXEL_MODE_COMPARISON:
    comparison_results_a = build_and_plot_both_voxel_modes(
        audio_results["a"]["full_energy_bins"],
        audio_results["a"]["full_active_mask"],
        energy_bins=ENERGY_BINS,
        title_prefix=f"{COMPARE_NAME_A}: {Path(audio_results['a']['audio_path']).name}",
        marker_size=PLOT_MARKER_SIZE,
        marker_opacity=PLOT_MARKER_OPACITY,
        template=PLOT_TEMPLATE,
    )
    comparison_results_b = build_and_plot_both_voxel_modes(
        audio_results["b"]["full_energy_bins"],
        audio_results["b"]["full_active_mask"],
        energy_bins=ENERGY_BINS,
        title_prefix=f"{COMPARE_NAME_B}: {Path(audio_results['b']['audio_path']).name}",
        marker_size=PLOT_MARKER_SIZE,
        marker_opacity=PLOT_MARKER_OPACITY,
        template=PLOT_TEMPLATE,
    )
else:
    print("Set RUN_VOXEL_MODE_COMPARISON = True in Cell 3 to run the optional surface-only voxel-mode comparison helpers.")


In [ ]:
# Cell 25 - Summary Footer

def export_summary_dataframe(dataframe, output_path, default_filename, description):
    """Export a summary dataframe to CSV if requested."""
    export_path = resolve_csv_export_path(default_filename, output_path=output_path)
    dataframe.to_csv(export_path, index=False)
    print(f"Saved {description} CSV to: {export_path}")
    return export_path


frame_duration_seconds = HOP_LENGTH / float(TARGET_SR)
summary_rows = []
for file_label, result in audio_results.items():
    num_frames_full_clip = int(result["band_energy"].shape[1])
    loaded_clip_duration_seconds = float(result["audio_duration_seconds"])
    voxelized_duration_seconds = num_frames_full_clip * frame_duration_seconds
    average_surface_points_per_frame = (
        result["total_surface_voxels"] / float(num_frames_full_clip)
        if num_frames_full_clip
        else 0.0
    )
    max_possible_surface_rows = result["voxel_volume"].shape[0] * result["voxel_volume"].shape[2]
    surface_density = (
        result["total_surface_voxels"] / float(max_possible_surface_rows)
        if max_possible_surface_rows
        else 0.0
    )
    max_audio_seconds_applied = MAX_AUDIO_SECONDS is not None
    was_truncated = (
        MAX_AUDIO_SECONDS is not None
        and loaded_clip_duration_seconds >= float(MAX_AUDIO_SECONDS) - frame_duration_seconds
    )
    summary_rows.append(
        {
            "file_label": file_label,
            "compare_name": result["surface_voxels_df"]["compare_name"].iloc[0] if not result["surface_voxels_df"].empty else file_label,
            "file_name": Path(result["audio_path"]).name,
            "sample_rate": result["sr"],
            "num_frames_full_clip": num_frames_full_clip,
            "frame_duration_seconds": round(frame_duration_seconds, 6),
            "voxelized_frame_range": f"[0, {max(num_frames_full_clip - 1, 0)}]",
            "voxelized_duration_seconds": round(voxelized_duration_seconds, 6),
            "loaded_clip_duration_seconds": round(loaded_clip_duration_seconds, 6),
            "max_audio_seconds_applied": max_audio_seconds_applied,
            "was_truncated": was_truncated,
            "num_bands": NUM_BANDS,
            "energy_bins": ENERGY_BINS,
            "voxel_mode": VOXEL_MODE,
            "total_surface_voxels": result["total_surface_voxels"],
            "average_surface_points_per_frame": round(average_surface_points_per_frame, 2),
            "surface_density": round(surface_density, 6),
            "inspection_view": PRIMARY_INSPECTION_VIEW,
            "normalization_enabled": NORMALIZE_BAND_ENERGY,
            "surface_csv_export": str(result["surface_voxel_csv_path"]) if result["surface_voxel_csv_path"] is not None else None,
        }
    )

summary_df = pd.DataFrame(summary_rows)
comparison_summary_df = pd.DataFrame(
    [
        {
            "compare_name_a": COMPARE_NAME_A,
            "compare_name_b": COMPARE_NAME_B,
            "surface_aligned_comparison_rows": len(surface_diff_df),
            "surface_changed_rows": int(surface_diff_df["changed"].sum()),
            "surface_mean_absolute_delta": round(float(surface_diff_df["abs_delta_energy_bin"].dropna().mean()), 4)
            if surface_diff_df["abs_delta_energy_bin"].notna().any()
            else np.nan,
            "surface_max_absolute_delta": round(float(surface_diff_df["abs_delta_energy_bin"].dropna().max()), 4)
            if surface_diff_df["abs_delta_energy_bin"].notna().any()
            else np.nan,
            "surface_difference_csv_export": str(surface_diff_csv_path) if surface_diff_csv_path is not None else None,
            "inspection_view": PRIMARY_INSPECTION_VIEW,
        }
    ]
)

summary_csv_path = None
comparison_summary_csv_path = None
if EXPORT_SUMMARY_CSV:
    summary_csv_path = export_summary_dataframe(
        summary_df,
        SUMMARY_CSV_OUTPUT_PATH,
        f"{export_timestamp}_summary_per_file.csv",
        "per-file summary",
    )
    comparison_summary_csv_path = export_summary_dataframe(
        comparison_summary_df,
        COMPARISON_SUMMARY_CSV_OUTPUT_PATH,
        f"{export_timestamp}_summary_comparison.csv",
        "comparison summary",
    )
else:
    print("EXPORT_SUMMARY_CSV is False; skipping summary CSV exports.")

print("Per-file summary")
display(summary_df)
print("Comparison summary")
display(comparison_summary_df)

if summary_csv_path is not None:
    print(f"Per-file summary CSV: {summary_csv_path}")
if comparison_summary_csv_path is not None:
    print(f"Comparison summary CSV: {comparison_summary_csv_path}")
